# Modelle kombinieren: Voting, Mittelung und Stacking

Ein einzelnes Modell sieht die Daten immer nur durch seine eigenen Annahmen und macht deshalb seine eigenen, typischen Fehler. *Ensemble-Methoden* kombinieren mehrere Modelle so, dass sich ihre einzelnen Fehler teilweise aufheben, und die kombinierte Vorhersage ist meist stabiler und treffsicherer als die jedes einzelnen Mitglieds. In diesem Notebook arbeitest du dich durch vier Wege, Klassifikatoren auf einem medizinischen Datensatz zu kombinieren, von einer einfachen Mehrheitsentscheidung bis zu einem trainierten Metamodell.

## Lernziele

Am Ende dieses Notebooks kannst du:

- ein Hard-Voting- und ein Soft-Voting-Ensemble mit dem `VotingClassifier` von scikit-learn bauen.
- Wahrscheinlichkeiten von Modellen per einfacher und gewichteter Mittelung kombinieren.
- einen Stacking-Klassifikator aufbauen, sowohl mit `StackingClassifier` als auch aus seinen einzelnen Schritten.
- ein Ensemble mit Accuracy, F1 und Recall evaluieren und erklären, was die Scores bedeuten.

## Warum Modelle kombinieren?

Hinter Ensembles steckt die Idee der *Weisheit der Vielen*: Wenn mehrere unabhängige Stimmen jeweils etwas besser sind als der Zufall, ist die Mehrheitsentscheidung verlässlicher als jede einzelne Stimme. Dasselbe gilt für Modelle, aber nur, wenn zwei Bedingungen erfüllt sind:

1. **Jedes Modell ist besser als zufälliges Raten.** Schwache, aber nützliche Modelle zu kombinieren hilft; Modelle zu kombinieren, die nicht besser sind als ein Münzwurf, hilft nicht.
2. **Die Modelle machen _unterschiedliche_ Fehler.** Wenn sich alle Modelle bei denselben Zeilen irren, ändert ihre Mittelung nichts. Erst durch Vielfalt (verschiedene Algorithmen, verschiedene Features oder verschiedene Teilmengen der Daten) können sich die Fehler aufheben.

Dieses Notebook baut vier Kombinationsstrategien auf, geordnet danach, wie viel sie über das Kombinieren lernen:

| Strategie | Wie die Mitglieder kombiniert werden | Was gelernt wird |
|---|---|---|
| Max Voting | Mehrheitsentscheidung über die vorhergesagten Labels | Nichts (feste Regel) |
| Mittelung | Mittelwert der vorhergesagten Wahrscheinlichkeiten | Nichts (feste Regel) |
| Gewichtete Mittelung | Gewichteter Mittelwert der Wahrscheinlichkeiten | Feste Gewichte, die du wählst |
| Stacking | Ein zweites Modell wird auf den Vorhersagen der Mitglieder trainiert | Die Kombination selbst |

## Setup

### Bibliotheken importieren

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

RSEED = 42

### Daten laden und aufteilen

Wir verwenden den Diabetes-Datensatz der Pima. Die Datei hat keine Kopfzeile, deshalb übergeben wir `header=None` und sprechen die Spalten über ihren Index an: Spalte `8` ist die Zielvariable (1 = Diabetes, 0 = kein Diabetes), die Spalten `0` bis `7` sind die Features.

Wir teilen mit `stratify=y` auf, damit Trainings- und Testdatensatz dieselben Klassenanteile behalten wie der gesamte Datensatz. Das ist hier wichtig, weil die Klassen unbalanciert sind und eine nicht stratifizierte Aufteilung in einem der beiden Datensätze zu wenige positive Fälle lassen könnte.

In [ ]:
# Diabetes-Daten importieren
df = pd.read_csv("data/pima-native-americans-diabetes.csv", header=None)
df.head(2)

In [ ]:
# Features und Zielvariable definieren und in Trainings- und Testdatensatz aufteilen
y = df[8]
X = df.drop(8, axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=RSEED
)

In [ ]:
y_train.shape, y_test.shape

In [ ]:
y_train.value_counts(), y_test.value_counts()

Bei der Aufteilung bleibt das Klassenungleichgewicht erhalten: Etwa zwei Drittel der Patientinnen sind negativ (kein Diabetes) und ein Drittel positiv. Behalte das im Kopf, wenn du später die Accuracy liest, denn ein Modell, das immer „kein Diabetes“ vorhersagen würde, käme schon auf etwa 65%.

### Baseline: Wie gut ist jedes Modell allein?

Bevor wir etwas kombinieren, messen wir jedes Modell für sich. Diese Baselines sind die Messlatte, die die Ensembles überspringen müssen: Ein Ensemble lohnt sich nur, wenn es sein bestes einzelnes Mitglied schlägt.

In [ ]:
# Jedes Modell für sich trainieren und die Test-Accuracy vergleichen
baselines = {
    "Logistic Regression": LogisticRegression(random_state=RSEED, max_iter=1000),
    "K-Nearest Neighbours": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=RSEED),
    "Random Forest": RandomForestClassifier(random_state=RSEED),
}

for name, single_model in baselines.items():
    single_model.fit(X_train, y_train)
    print(f"{name:22s} test accuracy: {single_model.score(X_test, y_test):.3f}")

Die vier Modelle landen auf dem Testdatensatz zwischen etwa 0,69 und 0,75, am stärksten ist der Random Forest mit etwa 0,75. Keines dominiert klar, und sie kommen auf sehr unterschiedliche Weise zu ihren Entscheidungen. Deshalb sind sie gute Kandidaten für eine Kombination.

## Max Voting

Beim *Max Voting (Hard Voting)* sagt jedes Modell ein Klassenlabel vorher, und das Ensemble gibt das Label zurück, das die meisten Modelle gewählt haben, wie bei einer Abstimmung per Handzeichen. Es ist das einfachste Ensemble: keine Wahrscheinlichkeiten, keine Gewichte, nur eine Mehrheit. Wir kombinieren eine logistische Regression, ein K-Nearest-Neighbours-Modell und einen Entscheidungsbaum.

```mermaid
flowchart TD
    D["Trainingsdaten"] --> M1["Modell 1<br/>Logistische Regression"]
    D --> M2["Modell 2<br/>KNN"]
    D --> M3["Modell 3<br/>Entscheidungsbaum"]
    M1 -->|"Vorhersage"| C["Nach einer festen Regel kombinieren<br/>(Mehrheitsentscheidung oder Mittelwert)"]
    M2 -->|"Vorhersage"| C
    M3 -->|"Vorhersage"| C
    C --> P["Finale Vorhersage"]
```

In [ ]:
from sklearn.ensemble import VotingClassifier

model1 = LogisticRegression(random_state=RSEED, max_iter=1000)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model = VotingClassifier(
    estimators=[("lr", model1), ("knn", model2), ("dt", model3)], voting="hard"
)
model.fit(X_train, y_train)
model.score(X_test, y_test)

Das ergibt etwa 0,75, eine kleine Verbesserung gegenüber jedem der drei abstimmenden Modelle für sich (0,69 bis 0,745). Hard Voting verwirft allerdings, wie *sicher* sich jedes Modell war: Ein Modell, das knapp über 50% lag, zählt genauso viel wie eines, das sich völlig sicher war. Genau hier setzt Soft Voting an.

### Soft Voting

*Soft Voting* mittelt die vorhergesagten Wahrscheinlichkeiten, statt Labels zu zählen, und wählt dann die Klasse mit der höchsten mittleren Wahrscheinlichkeit. Ein sicheres Modell zieht das Ergebnis deshalb stärker auf seine Seite als ein unsicheres. Die einzige Änderung ist, `voting="soft"` zu setzen.

In [ ]:
# Soft Voting: die vorhergesagten Wahrscheinlichkeiten statt der Labels mitteln
soft_model = VotingClassifier(
    estimators=[("lr", model1), ("knn", model2), ("dt", model3)], voting="soft"
)
soft_model.fit(X_train, y_train)
soft_model.score(X_test, y_test)

Soft Voting erreicht etwa 0,766 und liegt damit über Hard Voting, weil es die zusätzliche Information in den Wahrscheinlichkeiten nutzt, statt jedes Modell auf ein einzelnes Label zu reduzieren.

## Mittelung

Wir können Wahrscheinlichkeiten auch von Hand mitteln, ohne den `VotingClassifier`. Jedes Modell liefert für jede Klasse eine Wahrscheinlichkeit; wir bilden den Mittelwert über die Modelle und sagen die Klasse mit der höchsten mittleren Wahrscheinlichkeit vorher. Das ist die manuelle Version von Soft Voting und macht die Mechanik sichtbar.

In [ ]:
model1 = LogisticRegression(random_state=RSEED, max_iter=1000)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

pred1 = model1.predict_proba(X_test)
pred2 = model2.predict_proba(X_test)
pred3 = model3.predict_proba(X_test)

finalpred = (pred1 + pred2 + pred3) / 3
finalpred = np.argmax(finalpred.round(0), axis=1)
(y_test == finalpred).sum() / len(finalpred)

Die Mittelung erreicht etwa 0,766 und schlägt damit Max Voting (0,75) und jedes einzelne Modell. Das Zusammenlegen der Wahrscheinlichkeiten gleicht die übermäßig sicheren Fehler der einzelnen Modelle aus.

## Gewichtete Mittelung

Die einfache Mittelung behandelt jedes Modell als gleich vertrauenswürdig. Die *gewichtete Mittelung* gibt besseren Modellen ein größeres Stimmgewicht. Als Gewicht dient hier die Trainings-Accuracy jedes Modells, so normalisiert, dass sich die Gewichte zu eins summieren.

> **Achtung:** Eine Gewichtung nach der *Trainings*-Accuracy belohnt Modelle, die auf dem Trainingsdatensatz Overfitting zeigen (ein Entscheidungsbaum kann eine fast perfekte Trainings-Accuracy erreichen, ohne zu generalisieren). In der Praxis gewichtest du besser nach der Leistung auf einem separaten Validierungsdatensatz oder tunst die Gewichte, statt den Trainings-Scores zu vertrauen.

In [ ]:
model1 = LogisticRegression(random_state=RSEED, max_iter=1000)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

pred1 = model1.predict_proba(X_test)
pred2 = model2.predict_proba(X_test)
pred3 = model3.predict_proba(X_test)

acc1 = accuracy_score(y_train, model1.predict(X_train))
acc2 = accuracy_score(y_train, model2.predict(X_train))
acc3 = accuracy_score(y_train, model3.predict(X_train))

acc_sum = acc1 + acc2 + acc3

weight1 = acc1 / acc_sum
weight2 = acc2 / acc_sum
weight3 = acc3 / acc_sum

finalpred = pred1 * weight1 + pred2 * weight2 + pred3 * weight3
finalpred = np.argmax(finalpred.round(0), axis=1)
(y_test == finalpred).sum() / len(finalpred)

Der Score liegt bei etwa 0,766, praktisch genauso hoch wie bei der einfachen Mittelung, weil die drei Modelle auf dem Trainingsdatensatz eine ähnliche Accuracy erreichen und die Gewichte deshalb fast gleich ausfallen. Die Gewichtung hilft am meisten, wenn sich die Modelle in ihrer Qualität deutlich unterscheiden.

## Stacking

Voting und Mittelung kombinieren Modelle nach einer *festen* Regel. *Stacking* geht einen Schritt weiter: Es trainiert ein zweites Modell, das **Metamodell**, das lernt, wie sich die Vorhersagen der Basismodelle am besten kombinieren lassen. Jedes Basismodell gibt Vorhersagen aus; diese Vorhersagen werden zu den Features für das Metamodell, das die finale Antwort liefert.

```mermaid
flowchart TD
    D["Trainingsdaten"] --> M1["Basismodell 1<br/>Entscheidungsbaum"]
    D --> M2["Basismodell 2<br/>KNN"]
    D --> M3["Basismodell 3<br/>Random Forest"]
    M1 -->|"vorhergesagte Wahrscheinlichkeiten"| MM["Metamodell<br/>Logistische Regression"]
    M2 -->|"vorhergesagte Wahrscheinlichkeiten"| MM
    M3 -->|"vorhergesagte Wahrscheinlichkeiten"| MM
    MM --> P["Finale Vorhersage"]
```

### Stacking mit `StackingClassifier`

Der `StackingClassifier` von scikit-learn übernimmt den gesamten Prozess. Du übergibst die Basis-Estimators und den `final_estimator` (das Metamodell), und er trainiert die Schichten für dich. Intern verwendet er Kreuzvalidierung, sodass das Metamodell aus Out-of-Fold-Vorhersagen lernt und nicht aus Vorhersagen, die die Basismodelle schon während ihres eigenen Trainings gesehen haben.

In [ ]:
# Implementierung von Stacking in Scikit-Learn
from sklearn.ensemble import StackingClassifier

estimators = [
    ("dt", DecisionTreeClassifier(random_state=RSEED)),
    ("knn", KNeighborsClassifier()),
    ("rf", RandomForestClassifier(random_state=RSEED)),
]

clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
clf.fit(X_train, y_train).score(X_test, y_test)

Damit das Beispiel oben einfach bleibt, hat das Stacking-Modell, das wir erstellt haben, nur zwei Ebenen. Die Modelle **DecisionTree, KNN und RandomForest** werden auf **Ebene null** gebaut, ein **LogisticRegression**-Modell dagegen auf **Ebene eins**.

 

Hier erreicht Stacking etwa 0,745. Mit nur drei Basismodellen auf einem kleinen Datensatz schlägt es die einfache Mittelung nicht, und das ist häufig so: Stacking lohnt sich, wenn es viele unterschiedliche Basismodelle gibt und genug Daten, damit das Metamodell eine nützliche Kombination lernen kann.

### Stacking von Hand

Um zu sehen, was `StackingClassifier` intern macht, bauen wir dasselbe Schritt für Schritt nach. Die größte Sorge dabei ist ein **Datenleck**: Das Metamodell muss aus Vorhersagen auf Daten lernen, auf denen die Basismodelle *nicht* trainiert wurden, sonst lernt es aus zu optimistischen Vorhersagen und zeigt Overfitting. Wir teilen den Trainingsdatensatz deshalb in zwei Hälften auf: Die Basismodelle werden auf der ersten Hälfte trainiert, und wir sammeln ihre Vorhersagen auf der zweiten Hälfte, um damit das Metamodell zu trainieren.

Wir nehmen drei Basisklassifikatoren und einen finalen Estimator getrennt voneinander und initialisieren für jeden das zugehörige Objekt.

In [ ]:
# Basis-Estimators
dt = DecisionTreeClassifier(random_state=RSEED)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RSEED)

# Finaler Estimator
final_est = LogisticRegression()

Für die Stacking-Klassifikation teilen wir unseren Trainingsdatensatz in zwei Teile auf
1. Mit dem ersten Teil trainieren wir unsere Basis-Estimators  
2. Und mit dem zweiten Teil sagen wir mit den Basis-Estimators Wahrscheinlichkeiten vorher und trainieren den finalen Estimator auf diesen Wahrscheinlichkeiten  

In [ ]:
X_train_1, X_train_2, y_train_1, y_train_2 = train_test_split(
    X_train, y_train, stratify=y_train, random_state=RSEED
)

In [ ]:
# Alle Basis-Estimators auf der 1. Hälfte des Trainingsdatensatzes trainieren
dt_model = dt.fit(X_train_1, y_train_1)
knn_model = knn.fit(X_train_1, y_train_1)
rf_model = rf.fit(X_train_1, y_train_1)

# Dann sagen wir mit den Basis-Estimators die Wahrscheinlichkeiten für die zweite Hälfte des Trainingsdatensatzes vorher
dt_probab = dt_model.predict_proba(X_train_2)[:, 1]
knn_probab = knn_model.predict_proba(X_train_2)[:, 1]
rf_probab = rf_model.predict_proba(X_train_2)[:, 1]

Wir haben jetzt für jede Patientin in der zweiten Hälfte drei Zahlen: die Wahrscheinlichkeit für Diabetes laut jedem einzelnen Basismodell. Diese drei Spalten werden zu den Trainings-Features für das Metamodell.

In [ ]:
# Dann kombinieren wir alle Wahrscheinlichkeiten und bilden daraus Trainingsdaten (Wahrscheinlichkeiten) für den finalen Estimator
lr_X = pd.concat(
    [
        pd.DataFrame(dt_probab, columns=["dt"]),
        pd.DataFrame(knn_probab, columns=["knn"]),
        pd.DataFrame(rf_probab, columns=["rf"]),
    ],
    axis=1,
)

In [ ]:
lr_X

In [ ]:
# Den finalen Estimator auf den kombinierten Wahrscheinlichkeiten und den Werten der Zielvariable trainieren
final_est.fit(lr_X, y_train_2)

Für den Testdatensatz machen wir dasselbe wie beim Training
1. Wahrscheinlichkeiten mit den Basis-Estimators auf dem Testdatensatz vorhersagen
2. die Wahrscheinlichkeiten der Basis-Estimators zu einem Testdatensatz für den finalen Estimator kombinieren
3. mit dem finalen Estimator auf den Testdaten (Wahrscheinlichkeiten) vorhersagen

In [ ]:
dt_pred = dt_model.predict_proba(X_test)[:, 1]
knn_pred = knn_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict_proba(X_test)[:, 1]

comb_pred = pd.concat(
    [
        pd.DataFrame(dt_pred, columns=["dt"]),
        pd.DataFrame(knn_pred, columns=["knn"]),
        pd.DataFrame(rf_pred, columns=["rf"]),
    ],
    axis=1,
)

pred_final = final_est.predict(comb_pred)

Zum Schluss berechnen wir die Scores für den von Hand gebauten Stack. Dieses Mal schauen wir über die Accuracy hinaus, weil die Klassen unbalanciert sind und die Accuracy allein in die Irre führen kann.

In [ ]:
from sklearn.metrics import f1_score, recall_score

print(f"Accuracy: {accuracy_score(y_test, pred_final):.3f}")
print(f"F1 score: {f1_score(y_test, pred_final):.3f}")
print(f"Recall: {recall_score(y_test, pred_final):.3f}")

Auf dem Testdatensatz ergibt das eine Accuracy von etwa 0,77, aber einen F1-Score von ungefähr 0,63 und einen Recall von etwa 0,57. Der Unterschied ist wichtig: Ein Recall von 0,57 bedeutet, dass das Modell rund 43% der Patientinnen übersieht, die tatsächlich Diabetes haben. Bei einem medizinischen Screening ist das der teure Fehler, deshalb zeichnet die Accuracy allein ein zu rosiges Bild. Lies Ergebnisse auf unbalancierten Daten immer mit einer Metrik, die sich auf die positive Klasse konzentriert.

## Zusammenfassung

In diesem Notebook hast du:

- Baselines einzelner Modelle gemessen und sie als Messlatte für jedes Ensemble verwendet.
- Hard-Voting- und Soft-Voting-Ensembles mit `VotingClassifier` gebaut.
- vorhergesagte Wahrscheinlichkeiten per einfacher und gewichteter Mittelung kombiniert.
- einen Stacking-Klassifikator mit `StackingClassifier` aufgebaut und von Hand nachgebaut.
- die Methoden mit Accuracy, F1 und Recall verglichen und sie auf unbalancierten Daten interpretiert.

| Methode | Test-Accuracy |
|---|---|
| Bestes einzelnes Modell (Random Forest) | 0,750 |
| Max Voting | 0,750 |
| Soft Voting | 0,766 |
| Mittelung | 0,766 |
| Gewichtete Mittelung | 0,766 |
| Stacking (`StackingClassifier`) | 0,745 |

Auf diesem Datensatz brachte die Mittelung der Wahrscheinlichkeiten die größte Verbesserung, während sich Stacking mit so wenigen Basismodellen nicht gelohnt hat. Die allgemeine Erkenntnis bleibt: Ensembles helfen am meisten, wenn die Mitglieder für sich ordentlich sind und unterschiedliche Fehler machen, und erst die richtige Metrik, nicht nur die Accuracy, zeigt dir, ob der Gewinn echt ist.

## Quellen & weiterführende Literatur

- [**Scikit-learn: Ensemble Methods**](https://scikit-learn.org/stable/modules/ensemble.html): Benutzerhandbuch zu Voting, Stacking, Bagging und Boosting.
- [**Scikit-learn: VotingClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html): API-Referenz für Hard Voting und Soft Voting.
- [**Scikit-learn: StackingClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html): API-Referenz für die Stacked Generalization.
- [**The Random Forest Algorithm (MLU-Explain)**](https://mlu-explain.github.io/random-forest/): Eine visuelle Erklärung dafür, warum die Kombination vieler Modelle funktioniert.